## Objetivo

O objetivo da análise exploratória de dados (EDA) é compreender melhor o conjunto de dados, investigando sua estrutura, distribuição e possíveis padrões. Além disso, busca-se identificar relações entre variáveis, detectar outliers e avaliar quais características podem impactar positiva ou negativamente o desempenho dos modelos de predição para o target **SalePrice** (o preço final de um imóvel dado um conjunto de características).

In [ ]:
import sys 
import os 

sys.path.append(os.path.abspath(".."))

In [ ]:
import numpy as np 
import math
import matplotlib.pyplot as plt 
import pandas as pd
import seaborn as sns
from src.data_cleaning import data_cleaning
from src.data_collection import data_collection

In [ ]:
data_collection()

In [ ]:
df = pd.read_csv('../data/train.csv', sep=',')

In [ ]:
df = df.drop('Id', axis=1)

In [ ]:
df.head()

In [ ]:
df.info()

## 1. Distribuição do target SalePrice

Como primeira etapa, identificaremos o comportamento da distribuição dos dados do target 'SalePrice'. Um dado $x_i$ é um outlier se:

$$x_i < Q_1 - 1.5 \times \text{IQR}$$

ou 

$$x_i > Q_3 + 1.5 \times \text{IQR}$$

onde $\text{IQR}$ é a distância inter-quartil: 

$$\text{IQR} = Q_3 - Q_1$$

In [ ]:
Q1 = df['SalePrice'].quantile(0.25)
Q3 = df['SalePrice'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df[(df["SalePrice"] < limite_inferior) | (df["SalePrice"] > limite_superior)]

print("Q1: ", Q1)
print("Q3: ", Q3)
print("IQR: ", Q3 - Q1)

outliers

In [ ]:
sns.histplot(data=df, x='SalePrice', kde=True)

plt.axvline(limite_inferior, color='red', linestyle='--', label='Limite inferior')
plt.axvline(limite_superior, color='green', linestyle='--', label='Limite superior')
plt.axvline(df['SalePrice'].median(), color='blue', linestyle='-', label='Mediana')
plt.axvline(df['SalePrice'].mean(), color='orange', linestyle='-', label='Média')

plt.legend()
plt.show()

In [ ]:
df.boxplot(column='SalePrice', fontsize=12, grid=True)

Como observado tanto no histograma quanto no boxplot, a variável alvo **SalePrice** apresenta uma distribuição assimétrica à direita (right-skewed), caracterizada por uma cauda mais longa em direção aos valores mais altos. Isso indica que a maioria dos imóveis possui preços concetrados em faixas menores, enquanto alguns poucos apresentam valores significativamente elevados. 

Para reduzir essa assimetria e aproximar a distribuição de uma forma mais próxima da normal, é comum aplicar uma transformação logarítmica na variável alvo. Essa transformação comprime os valores mais altos e tende a tornar a distribuição mais simétrica, o que pode melhorar o desempenho e a estabilidade dos modelos de regressão que serão usados.

In [ ]:
sns.histplot(np.log1p(df['SalePrice']), kde=True)

## 2. Valores ausentes

Vamos analisar o comportamento dos dados ausentes no dataset. Para isso, vamos calcular a porcentagem de dados ausentes em cada uma das colunas.

O primeiro passo é identificar quais colunas possuem dados ausentes:

In [ ]:
missing_cols = df.columns[df.isna().sum() > 0]

missing_cols

E então calculamos a porcentagem relativa à aquela coluna:

In [ ]:
missing_percent = (df[missing_cols].isna().sum() / len(df) * 100).sort_values(ascending=False)

missing_percent

Para decidir como tratar esses dados, precisamos entendê-los. No arquivo README.md, na pasta data, há uma descrição breve (mas suficiente) do que cada uma dessas colunas significam. Ao observa-lo, concluímos que grande parte dos das ausentes não significam dados faltantes, mas sim ausência da característica na casa.

| Variável | Como tratar missing values |
|----------|----------------------------|
| PoolQC | A grande quanidade de dados ausentes representa casas que não possuem piscina, substituir por "None"|
| MiscFeature | A residência não possui características não explicadas pelas outras variáveis, substituir por "None"|
| Alley | A residência não possui acesso a beco, substituir por "None" | 
| Fence | A residência não possui cercas, substituir por "None" |
| MasVnrType | A residência não possui revestimento de alvenaria, substituir por "None" | 
| FireplaceQu | A residência não possui lareira, substituir por "None" |
| LotFrontage | Não há informação do comprimento da rua conectada ao lote; pode ser estimado pela mediana do bairro (Neighborhood) |
| GarageType | A residência não possui garagem, substituir por "None"|
| GarageYrBlt | Não há informação sobre data de construção da garagem, substituir pelo mesmo ano de criação da casa (YearBuilt)| 
| GarageFinish | A residência não possui garagem, substituir por "None"| 
|GarageQual| A residência não possui garagem, substituir por "None"| 
|BsmtExposure| A residência não possui porão, substituir por "None"|
|BsmtFinType1| A residência não possui acabamento no porão, substituir por "None"|
|BsmtFinType2| A residência não possui segundo tipo de acabamento no porão, substituir por "None"|
|BsmtQual| A residência não possui porão, substituir por "None"|
|BsmtCond | A residência não possui porão, substituir por "None"| 
| BsmntFinType1 | A residência não possui acabamento principal no porão, substituir por "None"|
|MasVnrArea| Se MasVnrType = None, então MasVnrArea = 0|
| Electrical | Poucos valores faltantes, substituir pela moda| 


Com base na análise acima, o pipeline de pré-processamento final foi implementado em src/data_cleaning.py.

In [ ]:
df = data_cleaning(df)

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

Como primeira etapa, vamos analisar o quanto as variáveis categóricas influenciam no nosso target SalePrice. Para isso, o primeiro passo é identificá-las:


In [ ]:
cat_cols = list(df.select_dtypes(include=["object", "category"]).columns)

for col in cat_cols:
    print(col, df[col].nunique())

Agora, vamos analisar com mais detalhes as categorias de cada uma:

In [ ]:
for col in cat_cols:
    print(col)
    print(df[col].unique())
    print()

## Variáveis ordinais

As variáveis: 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC' e 'HeatingQC' são variáveis ordinais, onde cada uma das siglas significam:

| Sigla | Significado| 
|-------|-----------|
|Ex | Excellent |
|Gd | Good |
|TA | Typical/Average|
|Fa| Fair| 
|Po | Poor |

Assim, a ordem natural é Po < Fa < TA < Gd < Ex. Dessa forma, podemos mapea-las em variáveis numéricas 

In [ ]:
qual_map = { 
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

ordinal_cols = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "HeatingQC"
]

for col in ordinal_cols:
    df[col] = df[col].map(qual_map)

df[ordinal_cols] = df[ordinal_cols].fillna(0).astype(float)

In [ ]:
cols = ordinal_cols
n_cols = 2
n_rows = math.ceil(len(cols)/n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*4))
axes = axes.flatten()  

for i, col in enumerate(cols):
    sns.boxplot(x=col, y='SalePrice', data=df, ax=axes[i])
    axes[i].set_title(f'SalePrice por {col}')

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
df['SalePrice'] = df['SalePrice'].astype(float)
df_corr = df[ordinal_cols + ['SalePrice']].dropna()

corr_values = df_corr.corr()['SalePrice'].sort_values(ascending=False)
print(corr_values)

In [ ]:
from scipy.stats import f_oneway

groups = [df[df['ExterCond']==cat]['SalePrice'] for cat in df['ExterCond'].unique()]
f_stat, p_val = f_oneway(*groups)
print('p-value:', p_val)

A variável 'BsmtExposure' também é ordinal e representa o nível de exposição do porão e suas siglas significam:

| Sigla | Significado |
|-------|-------------|
|No | Sem exposição (porão interno, sem janelas) |
|Mn | Exposição mínima (pouca luz/ventilação)| 
| Av | Exposição média (alguma luz/ventilação)|
| Gd | Exposiçã ótima/boa (muito ventilado ou iluminado)|
| Na/NaN | Sem porão |

In [ ]:
df["BsmtExposure"] = df["BsmtExposure"].map({"No":0, "Mn":1, "Av":2, "Gd":3}).fillna(0)

As variáveis 'BsmtFinType1' e 'BsmtFinType2' também são ordinais e relacionam-se à qualidade do porão. Suas siglas significam:

| Sigla | Significado |
|-------|--------------|
| Unf | Inacabado |
|LwQ | Acabamento de baixa qualidade (Low Quality)|
| Rec | Sala de recreação (adequado, mas simples)|
| BLQ | Basement Low Quality / parcialmente acabado|
| ALQ | Basement Average / bom acabamento| 
| GLQ | Basement Good / excelente acabamento (Good Living Quarters)|
| Na/NaN | Sem porão |

In [ ]:
bsmt_finish_map = {"Unf":0, "LwQ":1, "Rec":2, "BLQ":3, "ALQ":4, "GLQ":5}

df["BsmtFinType1"] = df["BsmtFinType1"].map(bsmt_finish_map).fillna(0)
df["BsmtFinType2"] = df["BsmtFinType2"].map(bsmt_finish_map).fillna(0)

A variável 'Functional' também é ordinal e representa a funcionalidade da casa. Suas siglas significam:

| Sigla | Significado | 
|-------|-------------|
| Sal | Severely irregular (planta muito ruim)| 
| Sev | Severely irregular| 
| Maj2 | Maior problema 2 (algum defeito estrutural ou planta estranha)| 
| Maj1 | Maior problema 1| 
| Mod | Moderadamente funcional| 
| Min2 | Pequeno problema 2| 
| Min1 | Pequeno problema 1| 
| Typ | Typical / Normal (sem problemas)|
| Na/NaN | Não informado |

In [ ]:
df["Functional"] = df["Functional"].map({
    "Sal":1, "Sev":2, "Maj2":3, "Maj1":4, "Mod":5,
    "Min2":6, "Min1":7, "Typ":8
}).fillna(0)

A variável 'GarageFinish' também é ordinal e representa a finalização da garagem. Suas siglas significam:

| Sigla | Significado |
|------|--------------|
| Unf | Não acabada | 
| RFn | Semi-acabada/Raw finish| 
| Fin | Totalmente acabada|
| Na/NaN | Sem garagem | 

In [ ]:
df["GarageFinish"] = df["GarageFinish"].map({"Unf":0, "RFn":1, "Fin":2}).fillna(0)

A variável 'Fence' também é ordinal e representa a qualidade da cerca. Suas siglas significam:

| Sigla | Significado| 
|-------|------------|
| MnWw | Madeira de baixo custo| 
| GdWo | Madeira boa|
| MnPrv | Minimamente construída, privativa| 
| GdPrv | Bem construída, privativa|
| Na/NaN | Sem cerca |

In [ ]:
df["Fence"] = df["Fence"].map({"MnWw":1, "GdWo":2, "MnPrv":3, "GdPrv":4}).fillna(0)

## Variáveis binárias 

A variável 'CentralAir' é uma variável binária, onde 

| Sigla | Significado | 
|-------|-------------|
| Y | Sim |
| N | Não |

In [ ]:
df["CentralAir"] = df["CentralAir"].map({"N": 0, "Y": 1})

As variáveis 'Street' e 'Alley' são binárias, e suas siglas são:

| Sigla | Significado |
|-------|-------------|
| Paved | Pavimentada | 
| Grvl| Cascalho | 
| None/NaN | Não informado|

In [ ]:
df["Street"] = df["Street"].map({"Grvl": 0, "Paved": 1}).fillna(0)
df["Alley"] = df["Street"].map({"Grvl": 0, "Paved": 1}).fillna(0)

As variáveis 'MSZoning', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1/Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RooftMatl', 'Exterior1st/Exterior2nd', 'MasVnrType', 'Heating', 'Electrical', 'MiscFeature', 'SaleType' e 'SaleCondition' são variáveis nominais e serão tratadas com one-hot encoding.

In [ ]:
nominal_cols = ["MSZoning", "LotShape", "LandContour", "Utilities", "LotConfig", "LandSlope",
                "Neighborhood", "Condition1", "Condition2", "BldgType", "HouseStyle",
                "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd",
                "MasVnrType", "Heating", "Electrical", "MiscFeature", "SaleType",
                "SaleCondition"]

df[nominal_cols] = df[nominal_cols].fillna("None")

In [ ]:
for col in nominal_cols:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=col, y='SalePrice', data=df)
    plt.xticks(rotation=45)
    plt.title(f'SalePrice por {col}')
    plt.show()

In [ ]:
from scipy.stats import f_oneway

for col in nominal_cols:
    groups = [df[df[col]==cat]['SalePrice'] for cat in df[col].unique()]
    f_stat, p_val = f_oneway(*groups)
    print(col, 'p-value:', p_val)

Agora, vamos avaliar quais características (features) são importantes para determinar o valor de uma casa. Para isso, vamos analisar a distribuição dos dados numéricos:

In [ ]:
df_num = df.select_dtypes(include=['float64', 'int64'])
df_num.head()

In [ ]:
df_num = df_num.drop("SalePrice", axis=1)

In [ ]:
df_num.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8);

Agora, investigaremos a relação dessas variáveis numéricas com o target SalePrice.

In [ ]:
cols = df_num.columns 
n_cols = 4
n_rows = math.ceil(len(cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))

axes = axes.flatten()

for i, col in enumerate(cols):
    sns.scatterplot(x=df[col], y=df["SalePrice"], ax=axes[i])
    axes[i].set_title(f"{col} vs SalePrice")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Agora, calcularemos a correlação linear entre "SalePrice" e as variáveis numéricas. Matematicamente, o panads calcula a correlação linear de Pearson, que mede o quanto duas variáveis têm relação linear.

$$
r = \frac{\text{Cov}(X, Y)}{\sigma_{X}\sigma_{Y}}
$$

onde:

- $\text{Cov}(X,Y)$ é a covariância entre as variáveis;
- $\sigma_{X}, \ \sigma_{Y}$ os desvios padrão.

O valor de $r$ varia entre $-1$ e $1$, onde:

|Valor| Interpretação|
| --- | -------------- |
|1| correlaçãom linear positiva perfeita|
|0| nenhuma corerlação linear| 
|-1| correção linear negativa perfeita|

In [ ]:
df["SalePrice_log"] = np.log(df["SalePrice"])

corr = df.corr(numeric_only=True)["SalePrice_log"].sort_values(ascending=False)

corr

As variáveis 'OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF' possuem alta correlação linear com o target 'SalePrice_log'.

In [ ]:
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 10))

sns.heatmap(
    corr,
    mask=mask,
    cmap="coolwarm",
    annot=False
)

plt.title("Matriz de correlação")
plt.show()

## Identificando multicolinearidade

A multicolinearidade ocorre quando duas ou mais variáveis preditoras de um modelo apresentam forte correlação entre si. Assim, uma variável pode ser perfeitamente (ou aproximadamente) explicada como combinação linear de outras variáveis. Em uma regressão linear múltipla, estimamos coeficientes no modelo, na forma matricial:

$$
\hat{\beta} = (X^{T}X)^{-1}X^{T}y
$$

Quando há multicolinearidade, $X_i$ e $X_j$ são quase paralelos:

$$
X_j \approx cX_i
$$

gerando grandes variações em $\hat{\beta}$ para pequenas mudanças em $X_j$ e $X_i$, o que reduz o desempenho do modelo. No nosso dataset, existem algumas colunas que são candidatas a serem exemplo de multicolinearidade pois possuem a mesma interpretação. Por exemplo, 'GarageArea' e 'GarageCars' são maneiras de medir o tamanho de uma garagem, mudando somente a unidade de medida. Para avaliarmos multicolinearidade entre as variáveis numéricas do dataset, medimos a correlação entre elas:

In [ ]:
corr = df.corr(numeric_only=True)["SalePrice_log"].abs().sort_values(ascending=False)
top = corr[:15].index

sns.clustermap(df[top].corr(), cmap="coolwarm")

In [ ]:
corr = df.corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

threshold = 0.8

collinear_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > threshold
]

collinear_pairs

### Interpretação:

- A coluna '1stFlrSF' é a área do primeiro andar, enquanto 'TotalBsmtSF' a área do porão. Essas duas variáveis são correlacionadas porque casas maiores tendem a ter porões maiores, não sendo necessário remoção dessas colunas para avaliar o modelo;

- A coluna 'TotRmsAbvGrd' representa o número de quartos, enquanto 'GrLivArea' a área habitável. Como 'GrLivArea'costuma ser mais informativa, removeremos 'TotRmsAbvGrd';

- A coluna 'GarageYrBlt' repreenta o ano em que a garagem foi construída, enquanto 'YearBuilt' o ano em que a casa foi construída. Normalmente a garagem é construída junto com a casa, então removeremos 'GarageYrBlt';

- A coluna 'GarageCars' representa o número de carros e 'GarageArea' a área da garagem, como elas medem a mesma coisa e 'GarageCars' é mais interpretável, removeremos 'GarageArea'.